# Instruction fine-tuning with Accelerate

This notebook follows `model_training/pytorch/01_test_multi_gpu.ipynb`: define training in a function, then run that function with `notebook_launcher`.

> Restart this kernel before the launch cell. Do not run CUDA device checks, load a model, or move tensors to CUDA in the notebook parent process. Each Accelerate worker creates its own data loader, model, optimizer, and CUDA state.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import importlib.util
import logging
import sys

import torch
import torch.nn.functional as F
from accelerate import Accelerator, notebook_launcher
from transformers import AutoModelForCausalLM

project_root = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'model_training').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Run this notebook from inside the deep_learning repository')

sys.path.insert(0, str(project_root))
from model_training.framework.data.post_train.instruction_tuning.interfaces import (
    create_sft_dataloaders,
    load_instruction_records,
    prepare_sft_data,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s | %(message)s')

In [ ]:
SFT_CONFIG = {
    'data_path': str(project_root / 'datasets/instruction_tuning/instruction-data.json'),
    'output_dir': str(project_root / 'execution_results/qwen_instruction'),
    'model_name': 'Qwen/Qwen2.5-3B-Instruct',
    'num_processes': 2,
    'mixed_precision': 'fp16',
    'epochs': 2,
    'batch_size': 1,
    'accumulation_steps': 4,
    'max_length': 1024,
    'learning_rate': 1e-5,
    'weight_decay': 0.1,
    'train_fraction': 0.85,
    'test_fraction': 0.10,
    'num_workers': 0,
}

print('[STAGE] configuration')
print(f"✓ model: {SFT_CONFIG['model_name']}")
print(f"✓ processes: {SFT_CONFIG['num_processes']} | precision: {SFT_CONFIG['mixed_precision']}")
print(f"✓ batch/GPU: {SFT_CONFIG['batch_size']} | accumulation: {SFT_CONFIG['accumulation_steps']}")
print(f"✓ output: {SFT_CONFIG['output_dir']}")

## 2. Validate the launch inputs

This check deliberately does not call `torch.cuda.*`; CUDA must first initialize inside the workers.

In [ ]:
print('[STAGE] validate launch inputs')
missing = [name for name in ('torch', 'transformers', 'accelerate') if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"Missing packages in this kernel: {', '.join(missing)}")

data_path = Path(SFT_CONFIG['data_path'])
if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

records = load_instruction_records(data_path)
if SFT_CONFIG['num_processes'] < 1:
    raise ValueError('num_processes must be at least one')
print(f"✓ records: {len(records)} | training split: {int(len(records) * SFT_CONFIG['train_fraction'])}")
print('✓ CUDA is still uninitialized in this notebook parent process')

## 3. Define the worker training function

Everything that needs CUDA is created inside this function, exactly as in the multi-GPU reference notebook.

In [ ]:
def train_instruction_sft(config: dict[str, object]) -> None:
    """Train the instruction-SFT model in one Accelerate worker."""
    accelerator = Accelerator(
        mixed_precision=str(config['mixed_precision']),
        gradient_accumulation_steps=int(config['accumulation_steps']),
    )

    records = load_instruction_records(Path(str(config['data_path'])))
    prepared_sft = prepare_sft_data(
        records,
        model_name=str(config['model_name']),
        train_fraction=float(config['train_fraction']),
        test_fraction=float(config['test_fraction']),
    )
    loaders = create_sft_dataloaders(
        prepared_sft,
        batch_size=int(config['batch_size']),
        max_length=int(config['max_length']),
        num_workers=int(config['num_workers']),
    )

    model = AutoModelForCausalLM.from_pretrained(str(config['model_name']))
    model.config.use_cache = False
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(config['learning_rate']),
        weight_decay=float(config['weight_decay']),
    )
    model, optimizer, train_loader = accelerator.prepare(model, optimizer, loaders.train)

    print(
        f"[WORKER {accelerator.process_index}] device={accelerator.device} "
        f"batches={len(train_loader)}",
        flush=True,
    )

    for epoch in range(int(config['epochs'])):
        model.train()
        total_loss = torch.zeros((), device=accelerator.device)
        total_batches = torch.zeros((), device=accelerator.device)
        optimizer.zero_grad()

        for input_ids, labels in train_loader:
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    logits = model(input_ids=input_ids).logits
                    loss = F.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        labels.reshape(-1),
                        ignore_index=-100,
                    )
                accelerator.backward(loss)
                optimizer.step()
                optimizer.zero_grad()

            total_loss += loss.detach().float()
            total_batches += 1

        global_loss = accelerator.reduce(total_loss, reduction='sum')
        global_batches = accelerator.reduce(total_batches, reduction='sum')
        if accelerator.is_main_process:
            if global_batches.item() == 0:
                raise RuntimeError('The training loader produced zero batches')
            print(
                f"✓ epoch {epoch + 1}/{config['epochs']} | "
                f"mean loss: {(global_loss / global_batches).item():.4f}",
                flush=True,
            )
        accelerator.wait_for_everyone()

    output_dir = Path(str(config['output_dir']))
    if accelerator.is_main_process:
        output_dir.mkdir(parents=True, exist_ok=True)
    accelerator.wait_for_everyone()

    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(
        output_dir,
        is_main_process=accelerator.is_main_process,
        save_function=accelerator.save,
        state_dict=accelerator.get_state_dict(model),
    )
    if accelerator.is_main_process:
        prepared_sft.tokenizer.save_pretrained(output_dir)
        print(f"✓ saved model and tokenizer: {output_dir}", flush=True)
    accelerator.wait_for_everyone()

## 4. Launch all workers

Run this only after restarting the kernel and executing the preceding cells in order.

In [ ]:
print('[STAGE] launch Accelerate workers', flush=True)
notebook_launcher(
    train_instruction_sft,
    args=(SFT_CONFIG,),
    num_processes=int(SFT_CONFIG['num_processes']),
    mixed_precision=str(SFT_CONFIG['mixed_precision']),
)
print('✓ all workers completed', flush=True)

## 5. Inspect saved artifacts

In [ ]:
output_dir = Path(SFT_CONFIG['output_dir'])
print(f"[STAGE] inspect saved artifacts | {output_dir}")
if not output_dir.is_dir():
    raise FileNotFoundError(f"Output directory not found: {output_dir}")
artifacts = sorted(path.name for path in output_dir.iterdir() if path.is_file())
print('✓ files:', artifacts)